# Cross-Dataset Alpha Combiner & Simulator

3개 이상의 데이터셋에서 0-fail 알파들을 GPT를 통해 조합하여
Non-PowerPool 알파를 생성하고 Brain API로 시뮬레이션을 수행합니다.

**Non-PowerPool 조건:**
- 3개 이상의 서로 다른 데이터셋 사용, 또는
- 2개 데이터셋 사용시 8개 이상의 연산자 사용

**허용된 조합 연산자:** add, subtract, min, max

## 1. Imports & Configuration

In [34]:
import json
import os
import re
import sys
import time
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from openai import OpenAI
from dotenv import load_dotenv

# Add current directory to path for ace_lib import
sys.path.insert(0, str(Path('.').resolve()))

import ace_lib as ace

In [35]:
# 경로 설정
SCRIPT_DIR = Path('.').resolve()
OUTPUT_FILE = SCRIPT_DIR / "combined_alpha.txt"
GOOD_ALPHA_FILE = SCRIPT_DIR / "good_alpha_list.json"

# 데이터셋 파일 목록 (확장 가능)
DATASET_FILES = {
    'mdl25': SCRIPT_DIR / 'mdl25.txt',
    'mdl30': SCRIPT_DIR / 'mdl30.txt',
    'mdl138': SCRIPT_DIR / 'mdl138.txt',
}

# 시뮬레이션 설정
REGION = "EUR"
UNIVERSE = "TOP2500"
DELAY = 1

# 허용된 조합 연산자
ALLOWED_OPERATORS = ['add', 'min', 'max']

print(f"SCRIPT_DIR: {SCRIPT_DIR}")
print(f"Dataset files: {list(DATASET_FILES.keys())}")

SCRIPT_DIR: C:\Users\adg01\llm_alpha_gen\llm_alpha_gen
Dataset files: ['mdl25', 'mdl30', 'mdl138']


## 2. 유틸리티 함수

In [36]:
def load_json(path: Path) -> list:
    """JSON 파일 로드"""
    if not path.exists():
        return []
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_json(data: list, path: Path):
    """JSON 파일 저장"""
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def append_to_json(new_items: list, path: Path):
    """JSON 파일에 항목 추가"""
    existing = load_json(path)
    existing.extend(new_items)
    save_json(existing, path)

## 3. 데이터셋 txt 파일 파싱

In [37]:
def parse_dataset_file(filepath: Path) -> List[Dict]:
    """
    데이터셋 txt 파일을 파싱하여 0-fail 알파들만 추출

    Returns:
        List[Dict]: [{'id': ..., 'expression': ..., 'sharpe': ..., 'fitness': ..., 'turnover': ...}, ...]
    """
    if not filepath.exists():
        print(f"[WARNING] File not found: {filepath}")
        return []

    content = filepath.read_text(encoding="utf-8")

    # Pattern to match each block (0-fail only)
    block_pattern = re.compile(
        r"--- #(\d+) \| FAIL: 0[^-]*---\n"
        r"ID: ([A-Za-z0-9]+)\n"
        r"Region: ([A-Z]+), Universe: ([A-Z0-9]+)\n"
        r"Sharpe: ([\d.-]+), Fitness: ([\d.-]+), Turnover: ([\d.-]+)\n"
        r"Expression: (.+?)\n",
        re.DOTALL
    )

    results = []
    for match in block_pattern.finditer(content):
        results.append({
            "block_num": int(match.group(1)),
            "id": match.group(2),
            "sharpe": float(match.group(5)),
            "fitness": float(match.group(6)),
            "turnover": float(match.group(7)),
            "expression": match.group(8).strip(),
        })

    return results

def load_all_zero_fail_alphas() -> Dict[str, List[Dict]]:
    """
    모든 데이터셋 파일에서 0-fail 알파들을 로드

    Returns:
        Dict[str, List[Dict]]: {'mdl25': [...], 'mdl30': [...], 'mdl138': [...]}
    """
    all_alphas = {}

    for dataset_name, filepath in DATASET_FILES.items():
        alphas = parse_dataset_file(filepath)
        all_alphas[dataset_name] = alphas
        print(f"[INFO] {dataset_name}: {len(alphas)} zero-fail alphas loaded")

    return all_alphas

## 4. GPT API를 통한 알파 조합 생성

In [38]:
def create_combination_prompt(all_alphas: Dict[str, List[Dict]], num_combinations: int = 10) -> str:
    """
    GPT에게 전달할 프롬프트 생성
    """
    # 각 데이터셋의 알파 요약 생성
    dataset_summaries = []
    for dataset_name, alphas in all_alphas.items():
        if not alphas:
            continue

        alpha_list = []
        for i, alpha in enumerate(alphas[:10]):  # 상위 10개만 사용
            alpha_list.append(f"  {i+1}. {alpha['expression']}")
            alpha_list.append(f"      (Sharpe: {alpha.get('sharpe', 'N/A')}, Fitness: {alpha.get('fitness', 'N/A')})")

        dataset_summaries.append(f"""
=== {dataset_name.upper()} Dataset ===
{chr(10).join(alpha_list)}
""")

    prompt = f"""You are a quantitative alpha researcher. Your task is to combine alphas from DIFFERENT datasets to create new, non-PowerPool alphas.

{chr(10).join(dataset_summaries)}

=== COMBINATION RULES ===
1. ALLOWED OPERATORS for combination: add, subtract, min, max (ONLY these 4!)
2. Each combined alpha MUST satisfy ONE of these conditions:
   - Use alphas from 3 OR MORE different datasets, OR
   - Use alphas from 2 datasets AND have 8 OR MORE total operators in the final expression
3. Pick ONE alpha expression from each dataset you use
4. You can nest operators, e.g., add(alpha1, subtract(alpha2, alpha3))
5. Do NOT modify the original alpha expressions - use them exactly as-is
6. The final expression must be syntactically valid FASTEXPR

=== OUTPUT FORMAT ===
Generate exactly {num_combinations} combined alphas. Return a JSON object with this structure:
{{{{
  "combinations": [
    {{{{
      "expression": "add(FULL_ALPHA_EXPR_FROM_MDL25, subtract(FULL_ALPHA_EXPR_FROM_MDL30, FULL_ALPHA_EXPR_FROM_MDL138))",
      "datasets_used": ["mdl25", "mdl30", "mdl138"],
      "idea": "Brief description of the investment thesis",
      "rationale_data": "Why these specific data sources complement each other",
      "rationale_operators": "Why these operators are appropriate for combining the signals"
    }}}},
    ... (repeat for all {num_combinations} alphas)
  ]
}}}}

=== ALPHA EXAMPLES ===
Good 3-dataset combination:
  add(ts_zscore(mdl25_vrv421_71v, 252), subtract(quantile(mdl30_new_psprise_pct_fy1_eps), zscore(winsorize(ts_backfill(vec_avg(mdl138_4idpc),60),std=4))))

Good 2-dataset combination with 8+ operators:
  add(add(ts_zscore(mdl25_vrv421_71v, 252), quantile(ts_zscore(mdl25_vrv421_91v, 63))), subtract(min(zscore(winsorize(ts_backfill(vec_avg(mdl138_4idpc),60),std=4)), ts_max_diff(ts_backfill(vec_avg(mdl138_4idpc),120),60)), max(ts_av_diff(ts_backfill(vec_avg(mdl138_3idpc),5),10), zscore(ts_decay_linear(ts_backfill(vec_avg(mdl138_4idpc),5), 20)))))

Generate {num_combinations} diverse and creative combinations now. Return ONLY the JSON object:"""

    return prompt

def call_gpt_for_combinations_streaming(prompt: str, model: str = "gpt-4o-mini") -> List[Dict]:
    """
    GPT API를 스트리밍으로 호출하여 알파 조합 생성 (실시간 출력)
    """
    load_dotenv(".env.local")

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    try:
        print(f"[INFO] Calling GPT API ({model}) with streaming...")
        print("=" * 80)
        print("🔄 GENERATING ALPHAS IN REAL-TIME...")
        print("=" * 80)
        
        stream = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": "You are a quantitative finance expert specializing in alpha generation. Return only valid JSON without markdown code blocks."
                },
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=8000,
            stream=True
        )

        # 스트리밍 출력 수집
        accumulated_text = ""
        
        print("\n📡 Receiving response from GPT...\n")
        
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                delta = chunk.choices[0].delta.content
                accumulated_text += delta
                
                # 실시간으로 받는 내용 일부 표시 (디버깅용)
                print(delta, end='', flush=True)
        
        print("\n\n" + "=" * 80)
        print("📥 Response received. Parsing JSON...")
        print("=" * 80 + "\n")
        
        # JSON 추출 (```json ... ``` 블록 처리)
        content = accumulated_text.strip()
        json_match = re.search(r'```(?:json)?\s*(.*?)\s*```', content, re.DOTALL)
        if json_match:
            content = json_match.group(1)
            print("[INFO] Removed markdown code blocks")
        
        # JSON 파싱
        try:
            result = json.loads(content)
            
            if 'combinations' in result:
                combinations = result['combinations']
                
                # 각 조합을 포맷팅하여 출력
                print(f"\n{'=' * 80}")
                print(f"✨ PARSED {len(combinations)} ALPHAS")
                print(f"{'=' * 80}\n")
                
                for i, combo in enumerate(combinations, 1):
                    print(f"{'─' * 80}")
                    print(f"✅ Alpha #{i}")
                    print(f"{'─' * 80}")
                    print(f"📊 Expression:")
                    expr = combo.get('expression', 'N/A')
                    if len(expr) > 100:
                        print(f"   {expr[:100]}...")
                        print(f"   {expr[100:200]}..." if len(expr) > 200 else f"   {expr[100:]}")
                    else:
                        print(f"   {expr}")
                    
                    print(f"\n🗂️  Datasets: {', '.join(combo.get('datasets_used', []))}")
                    print(f"\n💡 Idea: {combo.get('idea', 'N/A')}")
                    
                    data_rat = combo.get('rationale_data', 'N/A')
                    print(f"\n📈 Data Rationale:")
                    print(f"   {data_rat[:80]}{'...' if len(data_rat) > 80 else ''}")
                    
                    op_rat = combo.get('rationale_operators', 'N/A')
                    print(f"\n🔧 Operator Rationale:")
                    print(f"   {op_rat[:80]}{'...' if len(op_rat) > 80 else ''}")
                    print()
                
                return combinations
            else:
                print("[ERROR] Response doesn't contain 'combinations' key")
                print(f"Keys found: {list(result.keys())}")
                return []
                
        except json.JSONDecodeError as e:
            print(f"[ERROR] Failed to parse JSON: {e}")
            print(f"\nFirst 500 chars of content:")
            print(content[:500])
            print("\n...")
            print(f"\nLast 500 chars of content:")
            print(content[-500:])
            return []

    except Exception as e:
        print(f"[ERROR] GPT API call failed: {e}")
        import traceback
        traceback.print_exc()
        return []

def generate_combined_alphas(all_alphas: Dict[str, List[Dict]],
                             num_combinations: int = 10,
                             model: str = "gpt-4o-mini") -> List[Dict]:
    """
    GPT를 사용하여 조합된 알파 생성 (스트리밍)
    """
    prompt = create_combination_prompt(all_alphas, num_combinations)

    combinations = call_gpt_for_combinations_streaming(prompt, model)

    if not combinations:
        print("[ERROR] Failed to generate combinations")
        return []

    print(f"\n[INFO] Successfully generated {len(combinations)} valid combinations from GPT")

    return combinations

## 5. 조합된 알파 검증

In [39]:
def count_operators(expression: str) -> int:
    """표현식에서 연산자 개수 카운트"""
    operators = re.findall(r'([a-z_]+)\s*\(', expression, re.IGNORECASE)
    return len(operators)

def detect_datasets_used(expression: str) -> List[str]:
    """표현식에서 사용된 데이터셋 감지"""
    datasets = set()

    if 'mdl25' in expression.lower():
        datasets.add('mdl25')
    if 'mdl30' in expression.lower():
        datasets.add('mdl30')
    if 'mdl138' in expression.lower():
        datasets.add('mdl138')
    if 'nws17' in expression.lower():
        datasets.add('nws17')
    if 'star_eps' in expression.lower():
        datasets.add('star')

    return list(datasets)

def validate_combination(combination: Dict) -> Tuple[bool, str]:
    """
    조합이 Non-PowerPool 조건을 만족하는지 검증

    Returns:
        (is_valid, reason)
    """
    expression = combination.get('expression', '')
    datasets_used = detect_datasets_used(expression)
    operator_count = count_operators(expression)

    # 조건 1: 3개 이상 데이터셋 사용
    if len(datasets_used) >= 3:
        return True, f"Uses {len(datasets_used)} datasets ({', '.join(datasets_used)})"

    # 조건 2: 2개 데이터셋 + 8개 이상 연산자
    if len(datasets_used) >= 2 and operator_count >= 8:
        return True, f"Uses {len(datasets_used)} datasets with {operator_count} operators"

    return False, f"INVALID: {len(datasets_used)} datasets, {operator_count} operators"

## 6. 결과 저장

In [40]:
def save_combined_alphas_to_txt(combinations: List[Dict], filepath: Path):
    """조합된 알파들을 txt 파일로 저장"""
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write("COMBINED ALPHAS (Cross-Dataset, Non-PowerPool)\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total: {len(combinations)} combinations\n")
        f.write("=" * 80 + "\n\n")

        valid_count = 0
        for i, combo in enumerate(combinations, 1):
            is_valid, reason = validate_combination(combo)
            datasets = detect_datasets_used(combo.get('expression', ''))
            ops = count_operators(combo.get('expression', ''))

            status = "VALID" if is_valid else "INVALID"
            if is_valid:
                valid_count += 1

            f.write(f"--- #{i} | {status} | Datasets: {len(datasets)}, Operators: {ops} ---\n")
            f.write(f"Expression: {combo.get('expression', 'N/A')}\n")
            f.write(f"Datasets Used: {', '.join(datasets)}\n")
            f.write(f"Validation: {reason}\n")
            f.write(f"Idea: {combo.get('idea', 'N/A')}\n")
            f.write(f"Data Rationale: {combo.get('rationale_data', 'N/A')}\n")
            f.write(f"Operator Rationale: {combo.get('rationale_operators', 'N/A')}\n")
            f.write("\n")

        f.write("=" * 80 + "\n")
        f.write(f"Summary: {valid_count}/{len(combinations)} valid combinations\n")
        f.write("=" * 80 + "\n")

    print(f"[INFO] Saved {len(combinations)} combinations to {filepath}")

## 7. Brain API 시뮬레이션

In [41]:
def build_description(combination: Dict) -> str:
    """시뮬레이션용 description 생성 (메타데이터로 저장용)"""
    idea = combination.get('idea', 'Combined alpha from multiple datasets')
    rationale_data = combination.get('rationale_data', 'Multi-dataset combination for diversification')
    rationale_ops = combination.get('rationale_operators', 'Basic arithmetic combination')

    description = f"""Idea: {idea}
Rationale for data used: {rationale_data}
Rationale for operators used: {rationale_ops}"""

    return description

def build_alpha_config(expression: str) -> Dict:
    """시뮬레이션용 알파 설정 생성 (Brain API 스펙에 맞춤)"""
    return {
        "type": "REGULAR",
        "settings": {
            "instrumentType": "EQUITY",
            "region": REGION,
            "universe": UNIVERSE,
            "delay": DELAY,
            "decay": 0,
            "neutralization": "INDUSTRY",
            "truncation": 0.08,
            "pasteurization": "ON",
            "unitHandling": "VERIFY",
            "nanHandling": "OFF",
            "language": "FASTEXPR",
            "testPeriod": "P0Y0M0D",
            "visualization": False,
        },
        "regular": expression,
    }

def simulate_combinations(session, combinations: List[Dict]) -> List[Dict]:
    """
    조합된 알파들을 Brain API로 시뮬레이션
    """
    # 유효한 조합만 필터링
    valid_combinations = []
    for combo in combinations:
        is_valid, reason = validate_combination(combo)
        if is_valid:
            valid_combinations.append(combo)
        else:
            print(f"  [SKIP] {reason}")

    if not valid_combinations:
        print("[WARNING] No valid combinations to simulate")
        return []

    print(f"\n[INFO] Simulating {len(valid_combinations)} valid combinations...")

    # 알파 설정 목록 생성
    alpha_list = []
    for combo in valid_combinations:
        expression = combo.get('expression', '')
        config = build_alpha_config(expression)
        alpha_list.append(config)

    # 시뮬레이션 실행
    try:
        results = ace.simulate_alpha_list_multi(
            session,
            alpha_list,
            limit_of_concurrent_simulations=3,
            limit_of_multi_simulations=3,
            simulation_config={
                "check_submission": True,
                "get_pnl": False,
                "get_stats": False,
            }
        )

        # 결과 병합 (combination_info 추가)
        for i, result in enumerate(results):
            if i < len(valid_combinations):
                # combination_info에 description 포함
                combo_info = valid_combinations[i].copy()
                combo_info['description'] = build_description(valid_combinations[i])
                result['combination_info'] = combo_info

        return results

    except Exception as e:
        print(f"[ERROR] Simulation failed: {e}")
        import traceback
        traceback.print_exc()
        return []

def save_simulation_results(results: List[Dict], filepath: Path) -> List[Dict]:
    """시뮬레이션 결과를 good_alpha_list.json에 저장"""
    new_entries = []

    for result in results:
        # 결과가 None이거나 에러인 경우 스킵
        if not result or not isinstance(result, dict):
            print(f"[SKIP] Invalid result: {result}")
            continue

        # simulate_data가 없으면 스킵
        sim_data = result.get('simulate_data')
        if not sim_data or not isinstance(sim_data, dict):
            print(f"[SKIP] No simulate_data in result")
            continue

        is_data = sim_data.get('is', {})
        
        # 안전한 데이터 추출
        regular_data = sim_data.get('regular', {})
        expression = regular_data.get('code') if isinstance(regular_data, dict) else 'N/A'

        entry = {
            "timestamp": datetime.now().isoformat(),
            "alpha_id": result.get('alpha_id', 'unknown'),
            "expression": expression,
            "is_sharpe": is_data.get('sharpe', 0),
            "is_fitness": is_data.get('fitness', 0),
            "is_turnover": is_data.get('turnover', 0),
            "settings": sim_data.get('settings', {}),
            "checks": is_data.get('checks', []),
            "combination_info": result.get('combination_info', {}),
            "source": "combine_and_simulate"
        }

        new_entries.append(entry)

        # 결과 출력
        sharpe = entry['is_sharpe'] or 0
        fitness = entry['is_fitness'] or 0
        turnover = entry['is_turnover'] or 0
        print(f"  [{entry['alpha_id']}] Sharpe: {sharpe:.2f}, Fitness: {fitness:.2f}, Turnover: {turnover:.4f}")

    if new_entries:
        append_to_json(new_entries, filepath)
        print(f"\n[INFO] Saved {len(new_entries)} results to {filepath}")
    else:
        print(f"\n[WARNING] No valid results to save")

    return new_entries

## 8. 데이터셋 파일 관리

In [42]:
def add_dataset_file(dataset_name: str, filepath: str):
    """새로운 데이터셋 파일 추가 (런타임)"""
    global DATASET_FILES
    DATASET_FILES[dataset_name] = Path(filepath)
    print(f"[INFO] Added dataset: {dataset_name} -> {filepath}")

def list_dataset_files():
    """현재 등록된 데이터셋 파일 목록"""
    print("Registered dataset files:")
    for name, path in DATASET_FILES.items():
        exists = "OK" if path.exists() else "NOT FOUND"
        print(f"  - {name}: {path} [{exists}]")

# 데이터셋 파일 확인
list_dataset_files()

Registered dataset files:
  - mdl25: C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\mdl25.txt [OK]
  - mdl30: C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\mdl30.txt [OK]
  - mdl138: C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\mdl138.txt [OK]


## 9. 메인 파이프라인 실행

아래 셀에서 파라미터를 조정한 후 실행하세요.

In [43]:
# 파이프라인 파라미터 설정
NUM_COMBINATIONS = 100       # 생성할 조합 개수
GPT_MODEL = "gpt-4o-mini"   # 사용할 GPT 모델
RUN_SIMULATION = True       # Brain API 시뮬레이션 실행 여부

In [44]:
# Step 1: 데이터셋 로드
print("[Step 1] Loading zero-fail alphas from datasets...")
all_alphas = load_all_zero_fail_alphas()

total_alphas = sum(len(alphas) for alphas in all_alphas.values())
print(f"\nTotal alphas loaded: {total_alphas}")

if total_alphas == 0:
    print("[ERROR] No alphas loaded. Check dataset files.")

[Step 1] Loading zero-fail alphas from datasets...
[INFO] mdl25: 33 zero-fail alphas loaded
[INFO] mdl30: 20 zero-fail alphas loaded
[INFO] mdl138: 29 zero-fail alphas loaded

Total alphas loaded: 82


In [45]:
# Step 2: GPT로 조합 생성
print("[Step 2] Generating combinations using GPT...")
combinations = generate_combined_alphas(all_alphas, NUM_COMBINATIONS, GPT_MODEL)
print(f"\nGenerated {len(combinations)} combinations")

[Step 2] Generating combinations using GPT...
[INFO] Calling GPT API (gpt-4o-mini) with streaming...
🔄 GENERATING ALPHAS IN REAL-TIME...

📡 Receiving response from GPT...

{
  "combinations": [
    {
      "expression": "add(ts_zscore(mdl25_vrv421_71v, 252), subtract(quantile(mdl30_new_psprise_pct_fy1_eps), zscore(winsorize(ts_backfill(vec_avg(mdl138_4idpc),60),std=4))) ) )",
      "datasets_used": ["mdl25", "mdl30", "mdl138"],
      "idea": "Combine momentum from MDL25 with earnings surprise from MDL30 and volatility from MDL138.",
      "rationale_data": "MDL25 provides a strong momentum signal, MDL30 captures earnings quality, and MDL138 adds risk assessment.",
      "rationale_operators": "Addition captures overall bullishness, while subtraction filters for negative surprises."
    },
    {
      "expression": "add(subtract(zscore(winsorize(mdl25_vrv421_71v, std=3)), ts_max_diff(ts_backfill(vec_avg(mdl138_4idpc),120),60)), quantile(divide(star_eps_surprise_prediction_fy1, ts_std_de

In [46]:
# Step 3: 조합 검증
print("[Step 3] Validating combinations...")
valid_count = 0
for combo in combinations:
    is_valid, reason = validate_combination(combo)
    status = "OK" if is_valid else "FAIL"
    if is_valid:
        valid_count += 1
    print(f"  [{status}] {reason}")

print(f"\nValid combinations: {valid_count}/{len(combinations)}")

[Step 3] Validating combinations...
  [OK] Uses 3 datasets (mdl30, mdl138, mdl25)
  [OK] Uses 4 datasets (mdl30, star, mdl138, mdl25)
  [OK] Uses 3 datasets (mdl30, mdl138, mdl25)
  [OK] Uses 3 datasets (mdl30, mdl138, mdl25)
  [FAIL] INVALID: 2 datasets, 3 operators
  [OK] Uses 4 datasets (mdl30, star, mdl138, mdl25)
  [OK] Uses 3 datasets (mdl30, mdl138, mdl25)
  [OK] Uses 2 datasets with 15 operators
  [OK] Uses 3 datasets (mdl30, mdl138, mdl25)
  [OK] Uses 3 datasets (mdl30, mdl138, mdl25)
  [OK] Uses 3 datasets (mdl30, mdl138, mdl25)
  [OK] Uses 2 datasets with 8 operators
  [OK] Uses 4 datasets (mdl30, star, mdl138, mdl25)
  [OK] Uses 3 datasets (mdl30, star, mdl25)
  [OK] Uses 3 datasets (mdl30, mdl138, mdl25)
  [OK] Uses 4 datasets (mdl30, star, mdl138, mdl25)
  [OK] Uses 2 datasets with 8 operators
  [OK] Uses 3 datasets (mdl30, mdl138, mdl25)
  [OK] Uses 2 datasets with 15 operators
  [OK] Uses 3 datasets (mdl30, mdl138, mdl25)
  [OK] Uses 3 datasets (mdl30, star, mdl25)
  [O

In [47]:
# Step 4: txt 파일로 저장
print("[Step 4] Saving combinations to txt...")
save_combined_alphas_to_txt(combinations, OUTPUT_FILE)

[Step 4] Saving combinations to txt...
[INFO] Saved 50 combinations to C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\combined_alpha.txt


In [48]:
# Step 5: Brain API 시뮬레이션
if RUN_SIMULATION and valid_count > 0:
    print("[Step 5] Running Brain API simulation...")
    try:
        session = ace.start_session()
        simulation_results = simulate_combinations(session, combinations)

        if simulation_results:
            saved_entries = save_simulation_results(simulation_results, GOOD_ALPHA_FILE)
            print(f"\n[DONE] Simulated {len(simulation_results)} alphas, saved {len(saved_entries)} results")
    except Exception as e:
        print(f"[ERROR] Simulation failed: {e}")
        print("[TIP] Run simulation manually after authenticating Brain API")
else:
    print("[Step 5] Skipping simulation")
    simulation_results = []

[Step 5] Running Brain API simulation...
Complete biometrics authentication and press any key to continue: 
https://api.worldquantbrain.com/authentication/persona?inquiry=inq_1SrA98VJLbjX5qMZCPHatc4zqYKZ



KeyboardInterrupt: Interrupted by user

In [ ]:
# 결과 요약
print("=" * 60)
print("Pipeline Complete!")
print(f"  - Combinations generated: {len(combinations)}")
print(f"  - Valid (Non-PowerPool): {valid_count}")
print(f"  - Simulated: {len(simulation_results)}")
print(f"  - Output file: {OUTPUT_FILE}")
print("=" * 60)

Pipeline Complete!
  - Combinations generated: 0
  - Valid (Non-PowerPool): 0
  - Simulated: 0
  - Output file: C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\combined_alpha.txt
